# Multi-Agent 대표 패턴 학습 실습

---

## 학습 개요

### 학습 주제
- **Multi-Agent 시스템의 필요성**: 단일 Agent의 한계를 이해하고 Multi-Agent가 필요한 상황을 판단
- **Multi-Agent 대표 패턴**: Planner-Worker, Reflection 등 협업 패턴의 구조와 특징 학습
- **LangGraph 워크플로우 구성**: StateGraph를 활용한 에이전트 간 상태 공유 및 흐름 제어

### 학습 목표
- Multi-Agent 시스템의 구조를 이해하고, 상황에 적합한 패턴을 선택하여 구현할 수 있다
  - 단일 Agent의 한계를 설명하고 Multi-Agent가 필요한 상황을 판단할 수 있다
  - Planner-Worker 역할 분리 구조를 이해하고 구현할 수 있다
  - Reflection 패턴과 성과 평가를 통해 결과 품질을 향상시키고 정량적으로 확인할 수 있다
  - LangGraph StateGraph를 사용하여 Agent 간 상태 공유 워크플로우를 구성할 수 있다
  - Multi-Agent 대표 패턴 5가지의 특징과 적합한 상황을 비교 설명할 수 있다

### 핵심 개념
- **Multi-Agent 시스템**: 여러 Agent가 역할을 분담하여 협업하는 구조. 각 Agent는 독립적인 LLM 호출과 프롬프트를 가지며, State를 통해 정보를 공유한다. 단일 Agent 대비 복잡한 작업에서 품질과 신뢰성이 향상된다.
- **Planner Agent**: 사용자 요청을 분석하고 작업 계획을 수립하는 역할. "무엇을 해야 하는지"를 정의하며, 직접 실행하지 않고 계획만 수립한다. 계획의 명확성이 전체 결과 품질을 좌우한다.
- **Worker Agent**: Planner의 계획에 따라 실제 작업을 수행하는 역할. 계획의 각 단계를 구체적인 결과물로 변환한다. 전문화된 프롬프트로 각 단계에 집중한다.
- **Reflection 패턴**: 결과를 자기 검토하여 품질을 향상시키는 루프. Agent가 생성한 결과를 다시 평가하고 누락/오류를 보완한다. 원본 요청 충족 여부를 체계적으로 검토한다.
- **StateGraph**: LangGraph에서 제공하는 상태 기반 워크플로우 정의 클래스. 노드(Agent)와 엣지(흐름)로 구성되며, State 객체를 통해 Agent 간 정보를 전달한다. TypedDict를 상속하여 State 스키마를 정의한다.
- **성과 평가 (Reward Model)**: Multi-Agent 결과 품질을 정량적으로 측정하는 평가 함수. Planner 계획의 완성도, Worker 실행의 충실도, 전체 요청 충족도를 평가한다.

### 선행 지식
- Python 함수 정의, 클래스 기본 문법, 딕셔너리 조작 (`dict.get()`, `**dict` 언패킹)
- 4-2(1) 실습 완료 (ReAct Agent 구현 경험, Tool 사용 이해)
- LLM API 호출 방식 (`llm.invoke()`), 프롬프트 엔지니어링 기초
- LangChain 기본 사용법 (ChatOpenAI, 메시지 포맷)

---

## 실습 구성

### 학습 방향

- **실습 구성 방식**
  - 문제와 정답 코드가 병렬로 제공되며, 각 단계별 TODO 영역을 채우며 학습자가 직접 구현

- **Required Package**
  - `langchain>=0.3.0`: LLM 체인 프레임워크
  - `langchain-openai>=0.2.0`: OpenAI 모델 연동
  - `langgraph>=0.2.0`: Multi-Agent 워크플로우
  - `python-dotenv>=1.0.0`: 환경 변수 관리

- **실행 환경**
  - Python 3.10 이상 권장 (LangGraph의 타입 힌팅 기능 활용)

- **Step 요약**
  - **Step 1 (5분)**: 환경 설정 — 패키지 설치, API 키 설정
  - **Step 2 (10분)**: 단일 Agent의 한계 체감 — 복잡한 작업을 단일 Agent로 시도
  - **Step 3 (5분)**: Multi-Agent 패턴 개요 — 5가지 대표 패턴 학습
  - **Step 4 (15분)**: Planner-Worker 구현 — 계획 수립과 실행 분리
  - **Step 5 (15분)**: Reflection 패턴 추가 — 자기 검토 루프로 품질 향상
  - **Step 6 (10분)**: 성과 평가 — 결과 품질을 정량적으로 측정

### 문제 설명

- **문제 개요**: 이 실습은 Multi-Agent 시스템의 필요성을 이해하고, Planner-Worker 및 Reflection 패턴을 구현하기 위해 설계되었습니다. 학습자는 LangGraph StateGraph를 사용하여 에이전트 간 상태 공유 워크플로우를 구성하고, 최종적으로 Multi-Agent 대표 패턴 5가지의 특징과 적합한 상황을 비교 설명할 수 있어야 합니다.
- **요구사항 요약**
  - TODO 1: 단일 Agent로 복잡한 작업을 시도하여 한계 체감
  - TODO 2: Planner Node를 구현하여 단계별 계획 수립
  - TODO 3: Worker Node를 구현하여 계획을 실행
  - TODO 4: Reflection Node를 구현하여 결과 검토 및 개선
  - TODO 5: 성과 평가 함수를 구현하여 결과 품질 정량 측정
- **주의사항**
  - LLM API 키가 올바르게 설정되어야 합니다
  - 각 Agent의 프롬프트는 역할을 명확히 분리해야 합니다

---

# Step 1: 환경 설정

### Setup: 라이브러리 설치 및 API Key 설정

In [1]:
%%capture
# 필요한 패키지 설치
%pip install "langchain>=0.3.0" "langchain-openai>=0.2.0" "langgraph>=0.2.0" "python-dotenv>=1.0.0" -q

In [2]:
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 확인 (.env 미설정 시 직접 입력)
if not os.environ.get("GMS_KEY"):
    os.environ["GMS_KEY"] = input("🔑 GMS_KEY를 입력하세요: ")
    
print("✅ 환경 설정 완료")

🔑 GMS_KEY를 입력하세요:  S16P04A272-1447962b-cb9c-4ec6-bcbf-7667b98fc62b


✅ 환경 설정 완료


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.environ["GMS_KEY"],
    base_url="https://gms.ssafy.io/gmsapi/api.openai.com/v1/"
)

In [4]:
# 모델 연결 테스트
response = llm.invoke("3일 서울 여행 계획을 간단히 요약해줘.")
print("LLM 응답:")
print(response.content[:200] + "...")
print(f"\n✅ 모델 연결 확인 완료 ({llm.model_name})")

LLM 응답:
간단 요약(3일 기준)

Day 1 — 전통·역사
- 오전: 경복궁 관람 + 근정전/국립민속박물관
- 오후: 북촌 한옥마을 도보 산책 → 인사동 골동품·차 카페
- 저녁: 종로/광화문 근처 한식 저녁 또는 청계천 산책

Day 2 — 쇼핑·도심 전망
- 오전: 명동 쇼핑·거리 음식
- 오후: 동대문 DDP 또는 코엑스(몰) 방문
- 저녁: 남산 N서울타워 ...

✅ 모델 연결 확인 완료 (gpt-5-mini)


---

# Step 2: 단일 Agent의 한계 체감

### Concept Check: 단일 Agent의 특징과 한계

**왜 이 단계가 필요한가요?**  
Multi-Agent의 필요성을 이해하려면 먼저 단일 Agent의 한계를 직접 체감해야 합니다. 이론적인 설명보다 실제로 복잡한 작업을 단일 Agent에게 맡겨보고 결과의 문제점을 확인하는 것이 효과적입니다.

**단일 Agent**는 하나의 LLM이 모든 작업을 처리합니다:
- 장점: 구현이 간단함
- 단점: 복잡한 작업에서 혼란 발생, 작업 누락, 품질 저하

복잡한 작업 예시:
> "3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산도 포함해서."

이 작업은 여러 하위 작업을 포함합니다:
1. 여행 일정 계획
2. 맛집 추천
3. 관광지 추천
4. 예산 계산

#### 단일 Agent의 한계 시각화

```
┌─────────────────────────────────────────────────────────┐
│                    단일 Agent                            │
│  ┌─────────────────────────────────────────────────┐   │
│  │ 복잡한 요청                                       │   │
│  │ "3일 서울 여행 + 맛집 + 관광지 + 예산"            │   │
│  └─────────────────────────────────────────────────┘   │
│                         ↓                               │
│  ┌─────────────────────────────────────────────────┐   │
│  │ 하나의 LLM이 모든 작업을 한 번에 처리             │   │
│  │ - 일정 계획 + 맛집 + 관광지 + 예산 동시 처리      │   │
│  └─────────────────────────────────────────────────┘   │
│                         ↓                               │
│  ┌─────────────────────────────────────────────────┐   │
│  │ ⚠️ 발생 가능한 문제                               │   │
│  │ • 일부 요청 누락 (예: 예산 계산 생략)             │   │
│  │ • 정보 깊이 부족 (예: 맛집 가격대 누락)           │   │
│  │ • 일관성 문제 (예: 동선이 비효율적)               │   │
│  └─────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

단일 Agent가 이 모든 작업을 한 번에 처리하면 어떻게 될까요?

### TODO 1: 단일 Agent로 복잡한 작업 시도

- **요구사항**: 단일 Agent에게 복잡한 여행 계획 작업을 요청하고 결과를 확인합니다.
- **입력**: 
  - `complex_request` (str): 복잡한 여행 계획 요청 문자열
- **출력**: 
  - `response` (AIMessage): LLM의 응답 객체, `response.content`로 텍스트 접근
- **예상 결과**: 여행 계획이 출력되지만, 일부 정보 누락 또는 체계성 부족 가능
- **확인 방법**: `print(response.content)`로 출력 후 요청 사항 충족 여부 체크
- **힌트**: `llm.invoke()` 메서드를 사용합니다.

In [5]:
# TODO: 단일 Agent로 복잡한 작업 시도

# 복잡한 작업 요청
complex_request = """
3일 서울 여행 계획을 세워줘.
- 각 날짜별 상세 일정
- 점심, 저녁 맛집 추천 (위치, 메뉴, 가격대)
- 관광지 추천 (입장료, 소요시간)
- 전체 예산 계산
"""

# TODO: llm.invoke()를 사용하여 단일 Agent로 처리하세요
response = llm.invoke(complex_request)

if response is not None:
    print("=== 단일 Agent 응답 ===")
    print(response.content)
else:
    print("=== 코드를 완성해주세요 ===")

=== 단일 Agent 응답 ===
아래는 3일간의 서울 여행 일정(세부 시간표), 식당 추천(위치·대표메뉴·가격대), 관광지(입장료·소요시간)과 전체 예산 추정입니다. 원하시면 일정·음식 취향(한식/퓨전/채식 등)이나 숙소 위치(명동/홍대/강남 등)에 맞춰 맞춤 조정해 드릴게요.

요약(빠르게 보기)
- 일정: 1일차(경복궁 → 북촌 → 인사동 → 명동), 2일차(남산·N서울타워 → 이태원 → 전쟁기념관 → 홍대 밤거리), 3일차(봉은사 → COEX → 롯데월드타워·석촌호수 → 한강 야경)
- 추천 식사: 토속촌 삼계탕(경복궁 근처), 명동교자(명동), 메이플트리하우스(이태원·한우구이) 등
- 주요 입장료(예시): 경복궁 3,000원, N서울타워 전망대 약 11,000원, 롯데월드타워(Seoul Sky) 약 27,000원
- 전체 예산(1인, 3박 기준, 중간 수준 추정): 약 60만 원 내외 (숙소·식사·교통·입장료·여유비 포함). 아래에서 상세분해 제시.

-------------------------
1일차 (도심+전통) — 경복궁·북촌·인사동·명동
- 09:00 경복궁 도착 — 관람 및 경회루/근정전 주변 산책
  - 소요시간: 1.5–2시간
  - 입장료: 성인 3,000원 (학생·경로 할인 별도)
  - 팁: 오전 10시경 수문장 교대식 있는 날도 있으니 확인
  - 가는법: 경복궁역(3호선) 하차

- 11:30 북촌 한옥마을 산책
  - 소요시간: 1–1.5시간
  - 입장료: 무료(개별 한옥 체험은 유료)
  - 이동: 도보 15분 내외

- 13:00 점심 — 토속촌 삼계탕 (경복궁/북촌 근처)
  - 위치: 종로구 재동 (경복궁/안국역 근처)
  - 추천메뉴: 삼계탕(전통 인삼 닭죽) — 14,000–18,000원/인
  - 소요시간: 1시간

- 14:30 인사동 거리 & 전통 찻집
  - 소요시간: 1.5–2시간
  - 볼거리: 전통 공예품, 기념품, 찻집
  - 이동: 도보/버스

- 17:00 청계천 걷기(간단한 산책) → 명동 이

### Test: 단일 Agent의 한계 확인

위 결과를 확인해보세요:
- [ ] 모든 요청 사항이 포함되었나요? (일정, 맛집, 관광지, 예산)
- [ ] 정보가 체계적으로 정리되었나요? (날짜별, 카테고리별)
- [ ] 예산 계산이 정확한가요? (숙박+식비+교통+입장료 합계)
- [ ] 맛집과 관광지 정보가 구체적인가요? (위치, 가격, 소요시간)

**확인 기준**:
- 4가지 요청 사항 중 1개 이상 누락 → 단일 Agent 한계 체감
- 정보 간 연결성 부족 (예: 동선 고려 없는 일정) → 체계성 문제 확인

**예상 출력 예시** (한계가 드러나는 경우):
```
✓ 일정: "Day 1: 경복궁, 명동..." (있음)
✓ 관광지: "경복궁, 북촌한옥마을..." (있음)
△ 맛집: "광장시장 빈대떡" (가격대 누락)
✗ 예산: 전체 합계 없음 또는 부정확
✗ 동선: 강남→종로→잠실 (비효율적 이동)
```

**체감 포인트**: "하나의 Agent가 모든 걸 하려니 복잡하고, 일부 정보가 누락되거나 품질이 일정하지 않네"

---

# Step 3: Multi-Agent 패턴 개요

### Concept Check: Multi-Agent 시스템이란?

**왜 이 단계가 필요한가요?**  
Step 2에서 단일 Agent의 한계를 체감했으니, 이제 그 해결책인 Multi-Agent 시스템의 다양한 패턴을 학습합니다. 어떤 상황에 어떤 패턴이 적합한지 이해해야 실제 프로젝트에서 올바른 선택을 할 수 있습니다.

**Multi-Agent 시스템**은 여러 Agent가 역할을 나누어 협업하는 구조입니다.

#### Single Agent vs Multi-Agent

| 구분 | Single Agent | Multi-Agent |
|------|--------------|-------------|
| 구조 | 하나의 LLM이 모든 작업 처리 | 여러 LLM이 역할 분담 |
| 장점 | 구현 간단 | 복잡한 작업 처리 가능, 품질 향상 |
| 단점 | 복잡한 작업에서 한계 | 구현 복잡도 증가 |
| 적합한 상황 | 단순한 Q&A | 다단계 작업, 전문 영역 분리 필요 시 |

#### Multi-Agent 대표 패턴 5가지

| 패턴 | 설명 | 적합한 상황 | 난이도 |
|------|------|-------------|--------|
| **Planner-Worker** | 계획 수립 → 실행 분리 | 순차적 다단계 작업 | 하 |
| **Supervisor** | 관리자가 여러 Worker 조율 | 병렬 처리 필요한 독립 작업 | 중 |
| **Debate/Discussion** | 여러 Agent가 토론 후 결론 | 다양한 관점이 필요한 의사결정 | 중 |
| **Reflection** | 자기 검토/개선 루프 | 품질 보장이 중요한 작업 | 하 |
| **Hierarchical** | 계층적 위임 구조 | 대규모 복잡한 프로젝트 | 상 |

이번 실습에서는 **Planner-Worker**와 **Reflection** 패턴을 구현합니다.

#### LangGraph란?

**LangGraph**는 LangChain 팀이 개발한 워크플로우 오케스트레이션 라이브러리입니다:
- **StateGraph**: 상태 기반 그래프로 에이전트 흐름 정의. `TypedDict`를 상속하여 State 스키마 정의
- **Node**: 각 에이전트가 수행하는 작업 단위. `def node_func(state) -> state` 형태의 함수
- **Edge**: 노드 간 연결 (흐름 정의). `add_edge("node_a", "node_b")`로 순차 연결
- **State**: 에이전트 간 공유되는 상태 정보. 딕셔너리 형태로 전달되며 각 노드가 업데이트

#### LangGraph State 흐름도

```
┌──────────────────────────────────────────────────────────────────┐
│                     LangGraph State 흐름                          │
├──────────────────────────────────────────────────────────────────┤
│                                                                   │
│  [State 객체]                                                     │
│  ┌─────────────────────────────────────────────┐                 │
│  │ user_request: "3일 서울 여행..."             │                 │
│  │ plan: ""                                     │                 │
│  │ result: ""                                   │                 │
│  └─────────────────────────────────────────────┘                 │
│           │                                                       │
│           ▼                                                       │
│  ┌─────────────────┐                                             │
│  │  Planner Node   │ ─── State 읽기: user_request                │
│  │  (계획 수립)     │ ─── State 쓰기: plan = "Step 1..."         │
│  └─────────────────┘                                             │
│           │                                                       │
│           ▼ (Edge: planner → worker)                             │
│  ┌─────────────────┐                                             │
│  │  Worker Node    │ ─── State 읽기: user_request, plan          │
│  │  (실행)         │ ─── State 쓰기: result = "Day 1: ..."       │
│  └─────────────────┘                                             │
│           │                                                       │
│           ▼                                                       │
│  [최종 State]                                                     │
│  ┌─────────────────────────────────────────────┐                 │
│  │ user_request: "3일 서울 여행..."             │                 │
│  │ plan: "Step 1: 일정...\nStep 2: 맛집..."    │ ← Planner 결과  │
│  │ result: "Day 1: 경복궁(09:00)..."           │ ← Worker 결과   │
│  └─────────────────────────────────────────────┘                 │
│                                                                   │
└──────────────────────────────────────────────────────────────────┘
```

### Guided Build: LangGraph State 및 기본 구조

LangGraph를 사용하려면 먼저 **State(상태)**를 정의해야 합니다. 
State는 에이전트들이 공유하는 정보를 담는 컨테이너입니다.

In [6]:
# LangGraph import 및 State 정의
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class AgentState(TypedDict):
    """Multi-Agent 시스템의 상태 정의"""
    user_request: str     # 사용자 요청
    plan: str             # Planner가 생성한 계획
    result: str           # Worker가 생성한 결과
    reflection: str       # Reflection Agent의 검토 결과
    final_result: str     # 최종 결과
    reflection_count: int # Reflection 반복 횟수 (무한 루프 방지)
    messages: List[dict]  # 에이전트 간 메시지 기록

# State 구조 확인
print("AgentState 필드:")
print("  - user_request: 사용자 요청 문자열")
print("  - plan: Planner Agent의 계획")
print("  - result: Worker Agent의 실행 결과")
print("  - reflection: Reflection Agent의 검토 결과")
print("  - final_result: 최종 결과")
print("  - reflection_count: Reflection 반복 횟수 (무한 루프 방지)")
print("  - messages: 에이전트 간 메시지 기록")

AgentState 필드:
  - user_request: 사용자 요청 문자열
  - plan: Planner Agent의 계획
  - result: Worker Agent의 실행 결과
  - reflection: Reflection Agent의 검토 결과
  - final_result: 최종 결과
  - reflection_count: Reflection 반복 횟수 (무한 루프 방지)
  - messages: 에이전트 간 메시지 기록


/opt/conda/lib/python3.11/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


---

# Step 4: LangGraph로 Planner-Worker 구현

### Concept Check: 역할별 프롬프트 설계

**왜 이 단계가 필요한가요?**  
이제 Multi-Agent 패턴 중 가장 기본적인 Planner-Worker를 직접 구현합니다. "계획"과 "실행"을 분리함으로써 각 Agent가 자신의 역할에 집중할 수 있고, 결과의 체계성이 향상됩니다.

각 Agent는 고유한 역할과 프롬프트를 가집니다:

- **Planner**: 요청을 분석하고 단계별 계획을 수립. "무엇을", "어떤 순서로" 할지 정의
- **Worker**: 계획을 받아 각 단계를 실행. 구체적인 정보와 결과물 생성

LangGraph에서는 각 Agent를 **노드(Node)**로 정의하고, State를 통해 정보를 전달합니다.

#### Agent 간 메시지 흐름 예시
```
[User] "3일 서울 여행 계획"
    ↓
[Planner] State 읽기: user_request
    ↓ LLM 호출
[Planner] State 쓰기: plan = "Step 1: 일정 계획\nStep 2: 맛집 추천..."
    ↓
[Worker] State 읽기: user_request, plan
    ↓ LLM 호출
[Worker] State 쓰기: result = "Day 1: 경복궁(09:00-12:00)..."
```

### TODO 2: Planner Node 구현

- **요구사항**: 사용자 요청을 분석하고 단계별 계획을 수립하는 Planner Node를 구현합니다.
- **입력**: 
  - `state` (AgentState): 현재 상태 딕셔너리
    - `state["user_request"]` (str): "3일 서울 여행 계획..." 형태의 요청
- **출력**: 
  - `AgentState`: 업데이트된 상태 딕셔너리
    - `plan` (str): "Step 1: 일정 계획\nStep 2: 맛집 추천..." 형태의 계획
    - `messages` (List[dict]): 메시지 로그에 Planner 결과 추가
- **예상 결과**: 3~5개의 Step으로 구성된 계획 문자열
- **확인 방법**: `print(result_state['plan'])`으로 계획 출력 확인
- **힌트**: State를 받아서 업데이트된 State를 반환하는 함수로 구현합니다. `{**state, "plan": plan}`으로 State 업데이트

In [9]:
# TODO: Planner Node 구현

def planner_node(state: AgentState) -> AgentState:
    """
    Planner Node: 사용자 요청을 분석하고 단계별 계획을 수립합니다.
    
    Args:
        state: 현재 AgentState
    
    Returns:
        업데이트된 AgentState (plan 필드 추가)
    """
    user_request = state["user_request"]
    
    # TODO: Planner 프롬프트를 작성하세요
    # - 사용자 요청을 분석하고 단계별 계획을 수립하도록 지시
    # - "계획만 수립하고, 직접 실행하지 말 것"을 명시
    # - 출력 형식: Step 1: [작업], Step 2: [작업], ...
    planner_prompt = f"""
당신은 작업 계획을 수립하는 Planner Agent입니다.

사용자 요청을 분석하고, 수행해야 할 작업을 단계별로 나열하세요.
각 단계는 구체적이고 실행 가능해야 합니다.

중요: 계획만 수립하세요. 직접 실행하지 마세요.

출력 형식:
Step 1: [작업 설명]
Step 2: [작업 설명]
...

---
사용자 요청: {user_request}
"""
    
    # LLM 호출
    response = llm.invoke(planner_prompt)
    plan = response.content
    
    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Planner",
        "receiver": "Worker",
        "content": plan,
        "task_type": "plan"
    })
    
    # TODO: State 업데이트하여 반환
    return {
        **state,
        "plan": plan,  # TODO: plan 변수로 교체
        "messages": messages
    }

# 테스트
test_state = {
    "user_request": "3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산 포함.",
    "plan": "",
    "result": "",
    "messages": []
}

# TODO 완료 후 테스트
result_state = planner_node(test_state)
if result_state['plan'] is not None:
    print("=== Planner Node 결과 ===")
    print(f"계획:\n{result_state['plan']}")
else:
    print("=== 코드를 완성해주세요 ===")

=== Planner Node 결과 ===
계획:
Step 1: 사용자 기본정보 수집
- 요청자에게 여행 날짜(출발/복귀), 인원수(성인/어린이), 출발지(해외/국내), 숙박 희망지역(예: 명동/홍대/강남 등)을 질문.
- 선호 여행스타일(느긋한 관광 vs 빽빽한 일정), 관심사(역사/한옥·궁궐, 쇼핑, 맛집 탐방, 전통시장, 자연공원, 야경 등), 식사 제약(채식·알레르기) 확인.
- 예산 범위(총 예산/1인당 대략), 숙박 등급 선호(게스트하우스~5성급) 확인.
- 산출물: 사용자 응답을 정리한 요약표.

Step 2: 항공/교통 및 일정 기본조건 확보
- 도착/출발 시간(비행기/기차 시간)을 확인하고, 도심 이동 수단(공항철도/리무진/택시) 선호 확인.
- 숙박 체크인·체크아웃 시간 요구사항 확인.
- 산출물: 여행 기간 내 시간 제약(도착/출발 시간 포함) 목록.

Step 3: 관광지 후보 목록 조사(사용자 관심사 기반)
- 사용자가 선택한 관심사에 맞춰 서울의 주요 명소 후보(각 일자에 적합한 장소 포함)를 지역별로 10~15곳 선별.
- 각 명소에 대해 간단한 정보(영업시간, 휴무일, 입장료(유무), 소요 시간 예측)를 수집할 것.
- 산출물: 명소 후보 리스트(우선순위 표기 포함).

Step 4: 맛집 후보 조사(아침/점심/저녁 별)
- 사용자의 식성·예산을 반영해 지역별로 식당 후보를 각 끼니당 최소 3곳 이상 선정(현지 인기 맛집 + 가격대 표기).
- 각 식당에 대해 주소, 영업시간, 예약 필요 여부, 평균 1인 식사비용 추정, 메뉴 특징을 기록.
- 산출물: 끼니별 맛집 후보 목록(예약 권장도 표기).

Step 5: 동선 최적화 및 일별 안배 설계
- 명소와 맛집을 지도를 기반으로 지리적으로 묶어 하루 권장 동선(오전·점심·오후·저녁) 초안 작성.
- 각 구간별 이동수단(도보/지하철/버스/택시), 소요 시간(이동+관람) 및 휴식시간 포함.
- 산출물: 3일 분량의 시간대별 초안 일정(각 일자별 타임라인).

Step 6: 예산 산출 방

### TODO 3: Worker Node 구현

- **요구사항**: Planner의 계획을 받아 각 단계를 실행하는 Worker Node를 구현합니다.
- **입력**: 
  - `state` (AgentState): 현재 상태 딕셔너리
    - `state["plan"]` (str): Planner가 생성한 계획
    - `state["user_request"]` (str): 원본 사용자 요청
- **출력**: 
  - `AgentState`: 업데이트된 상태 딕셔너리
    - `result` (str): 계획의 각 단계를 실행한 구체적인 결과
    - `messages` (List[dict]): 메시지 로그에 Worker 결과 추가
- **예상 결과**: 각 Step에 대한 구체적인 정보가 포함된 결과물 (날짜별 일정, 맛집 목록 등)
- **확인 방법**: `print(final_state['result'][:500])`으로 결과 일부 확인
- **힌트**: State에서 plan을 읽어 실행하고 result를 업데이트합니다.

In [10]:
# TODO: Worker Node 구현

def worker_node(state: AgentState) -> AgentState:
    """
    Worker Node: Planner의 계획을 받아 각 단계를 실행합니다.
    
    Args:
        state: 현재 AgentState (plan 포함)
    
    Returns:
        업데이트된 AgentState (result 필드 추가)
    """
    plan = state["plan"]
    user_request = state["user_request"]
    
    # TODO: Worker 프롬프트를 작성하세요
    # - Planner의 계획을 받아 각 단계를 실행하도록 지시
    # - 각 단계별 구체적인 정보를 포함하도록 요청
    worker_prompt = f"""
당신은 계획을 실행하는 Worker Agent입니다.

Planner가 수립한 계획을 받아 각 단계를 실행하고 결과를 제공하세요.
각 단계별로 구체적인 정보를 포함하세요.

---
원본 요청: {user_request}

실행할 계획:
{plan}

---
위 계획을 단계별로 실행하고, 각 단계의 결과를 자세히 작성하세요.
"""
    
    # LLM 호출
    response = llm.invoke(worker_prompt)
    result = response.content

    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Worker",
        "receiver": "User",
        "content": result,
        "task_type": "result"
    })
    
    # TODO: State 업데이트하여 반환
    return {
        **state,
        "result": result,  # TODO: result 변수로 교체
        "messages": messages
    }

# TODO 완료 후 테스트
final_state = worker_node(result_state)
if final_state['result'] is not None:
    print("=== Worker Node 결과 ===")
    print(f"실행 결과:\n{final_state['result'][:500]}...")
else:
    print("=== 코드를 완성해주세요 ===")

=== Worker Node 결과 ===
실행 결과:
좋습니다. 저는 지금부터 Planner가 만든 실행계획을 따라 Worker Agent 역할로 진행하겠습니다. 먼저 Step 1(사용자 기본정보 수집)을 시작해야 다음 단계(교통·일정·맛집 조사 등)를 진행할 수 있습니다. 진행 허가를 주시면 아래 질문들을 드리고, 응답을 토대로 각 단계를 순차적으로 수행하여 결과(명소·맛집 후보, 동선안, 예산표, 예약 체크리스트 등)를 자세히 제공하겠습니다.

먼저 Step 1 질문 목록(복사·붙여넣기 편하게 정리) — 가능한 한 정확히 알려주세요:
1) 여행 날짜
   - 출발일(YYYY-MM-DD) / 복귀일(YYYY-MM-DD)  
   - 항공/기차/버스 시간(이미 예약되어 있으면 편명·시간 기재)
2) 인원수
   - 성인 몇 명, 어린이(또는 유아) 몇 명(연령 포함)
3) 출발지
   - 해외(국가/도시) 또는 국내(도시/역/공항)
4) 숙박 희망지역(예: 명동 / 홍대 / 강남 / 북촌 등) — 복수 선택 가능
5) 숙박 등급 선호
...


### Guided Build: LangGraph 워크플로우 구성

이제 Planner Node와 Worker Node를 **StateGraph**로 연결하여 워크플로우를 구성합니다.

In [11]:
from langgraph.graph import StateGraph, START, END

# LangGraph 워크플로우 정의 (Guided Build - 코드 제공)
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("planner", planner_node)
workflow.add_node("worker", worker_node)

# 엣지 연결: planner → worker → END
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "worker")
workflow.add_edge("worker", END)

# 컴파일
app = workflow.compile()

print("LangGraph 워크플로우 구성 완료!")
print("흐름: planner → worker → END")

LangGraph 워크플로우 구성 완료!
흐름: planner → worker → END


### Test: LangGraph Multi-Agent 실행

컴파일된 워크플로우를 실행하여 Planner → Worker 협업을 테스트합니다.

**확인 기준**:
- [ ] Planner가 3~5개의 명확한 Step을 생성했는가?
- [ ] Worker가 각 Step을 구체적인 정보로 실행했는가?
- [ ] State를 통해 plan이 Worker에게 전달되었는가?
- [ ] 단일 Agent 결과 대비 체계성이 향상되었는가?

**Agent 간 메시지 흐름 확인**:
```python
# messages 리스트에서 흐름 확인
for msg in result["messages"]:
    print(f"[{msg['sender']}] → [{msg['receiver']}]: {msg['task_type']}")
# 예상 출력:
# [Planner] → [Worker]: plan
# [Worker] → [User]: result
```

**체감 포인트**: "LangGraph로 계획과 실행을 분리하니 워크플로우가 명확하고 결과 품질도 높아졌네!"

In [12]:
# LangGraph 워크플로우 실행
def run_multi_agent(user_request: str) -> dict:
    """
    LangGraph Multi-Agent 시스템 실행
    
    Args:
        user_request: 사용자 요청
    
    Returns:
        최종 상태를 담은 딕셔너리
    """
    print("🚀 LangGraph Multi-Agent 시스템 시작\n")
    
    # 초기 상태 설정
    initial_state = {
        "user_request": user_request
    }
    
    # 워크플로우 실행
    print("📋 [Planner Agent] 계획 수립 중...")
    print("⚙️ [Worker Agent] 계획 실행 중...")
    
    final_state = app.invoke(initial_state)
    
    print("✅ 실행 완료\n")
    
    return final_state

# 테스트 실행
workflow_result = run_multi_agent("서울 2박3일 여행 계획 + 예산 50만원 이내")

print("=" * 50)
print("📊 최종 결과")
print("=" * 50)
print(workflow_result["result"])

🚀 LangGraph Multi-Agent 시스템 시작

📋 [Planner Agent] 계획 수립 중...
⚙️ [Worker Agent] 계획 실행 중...
✅ 실행 완료

📊 최종 결과
알겠습니다. 저는 Worker Agent로서 Planner가 만든 15단계 수행 계획을 하나씩 실행(안)하고, 각 단계의 결과·권고·다음 액션을 자세히 제시하겠습니다.

먼저 중요한 확인사항(필수 입력)이 없습니다: 출발지(거주 도시), 출발/도착 날짜(yyyy-mm-dd), 동행 인원 및 연령대(성인/청소년/유아), 이동 수단 선호(자가/기차/버스/비행기) 등. 실제 예약·확정은 사용자의 응답(또는 직접 예약 허가)이 있어야 진행 가능합니다. 아래는 두 가지 처리 방식 제안입니다 — (A) 사용자가 정보를 주시기 전까지 진행 가능한 “가정 기반 시안”을 만들어 단계별 실행 내용과 예약 우선순위를 제시하거나, (B) 우선 필수정보를 요청하여 그에 맞춰 바로 실예약(또는 구체 검색) 준비를 진행합니다.

먼저, 사용자의 확인을 빨리 받을 수 있도록 필수 질문을 적습니다. 동시에 요청하신 15단계 전부에 대해 “가정 기반 시안(모범안)”을 만들어 실행 결과(추정치·권장사항)를 제공하겠습니다. 사용자가 정보를 주시면 그 시안에 따라 실조정·예약 진행 가능함을 알려드립니다.

필수 확인(빠른 회신 부탁)
- 출발지(거주 도시): ?
- 여행 날짜(출발일 / 귀가일, yyyy-mm-dd): ?
- 총 인원 및 연령대(예: 성인 2명 / 유아 1명 등): ?
- 이동수단 선호(자가/고속버스/기차(KTX)/항공): 선택 또는 무관?
- 숙소 타입 선호(게스트하우스/비즈니스호텔/호텔/에어비앤비): ?
- 취향(아래 Step 2에서 다시 묻겠습니다) — 우선 임의 가정으로 진행 가능.

가정 기반 시안 전제(이후 모든 수치·권고는 이 가정하에 작성됨)
- 인원: 성인 1명
- 출발지: 부산 (대한민국 내 타지역일 경우 교통비·시간 재계산 필요)
- 일정: 2026-09-10(금) 출발 /

---

# Step 5: Reflection 패턴 추가

### Concept Check: Reflection 패턴이란?

**왜 이 단계가 필요한가요?**  
Planner-Worker만으로도 체계적인 결과를 얻을 수 있지만, 결과의 품질을 한 단계 더 높이려면 자기 검토가 필요합니다. Reflection 패턴은 "결과가 요청을 충족하는가?"를 스스로 평가하고 개선하는 **루프**입니다.

**Reflection 패턴**은 Agent가 자신의 결과를 검토하고 개선하는 자기 검토 루프입니다.

#### Reflection 루프 구조 (핵심)

```
[Planner] → [Worker] → [Reflection] ─── 품질 충분? ──→ [END]
                ↑                              │
                └──── 개선 필요 (재실행) ────────┘
```

**핵심 포인트**: Reflection은 단순 평가가 아니라 **평가→개선 루프**입니다.
- Reflection Agent가 결과를 검토하고 "충족/부분 충족/미충족"을 판정
- "부분 충족" 또는 "미충족"이면 개선된 결과를 만들어 Worker에게 다시 보냄
- 무한 루프 방지를 위해 `reflection_count`로 최대 반복 횟수를 제한 (예: 최대 1회)

#### Reflection Agent의 역할
- 결과가 원본 요청을 충족하는지 검토
- 누락된 정보나 개선 사항 식별
- **충족 시**: 최종 결과 확정 → END
- **미충족 시**: 개선 피드백을 result에 반영 → Worker 재실행

#### Agent 간 메시지 흐름 (Reflection 포함)
```
[Worker] State 쓰기: result = "Day 1: 경복궁..."
    ↓
[Reflection] State 읽기: user_request, plan, result
    ↓ LLM 호출 (검토)
[Reflection] 판정:
    충족 → State 쓰기: final_result = result → END
    부분 충족 → State 쓰기: result = "개선된 결과..." → Worker 재실행
```

### TODO 4: Reflection Agent 추가

- **요구사항**: Worker의 결과를 검토하고 개선 제안을 하는 Reflection Node를 구현합니다.
- **입력**: 
  - `state` (AgentState): 현재 상태 딕셔너리
    - `state["result"]` (str): Worker가 생성한 결과
    - `state["user_request"]` (str): 원본 사용자 요청
    - `state["plan"]` (str): Planner가 생성한 계획
- **출력**: 
  - `AgentState`: 업데이트된 상태 딕셔너리
    - `reflection` (str): 검토 결과 (충족 여부, 강점, 개선 필요)
    - `final_result` (str): 개선된 최종 결과
    - `messages` (List[dict]): 메시지 로그에 Reflection 결과 추가
- **예상 결과**: 
  - 검토 결과: "충족/부분 충족/미충족" 판정 + 구체적 피드백
  - 최종 결과: Worker 결과를 보완한 완성도 높은 결과물
- **확인 방법**: `print(reflection_state['reflection'][:800])`으로 검토 결과 확인
- **힌트**: 결과를 검토하고 누락된 부분이나 개선점을 제안합니다. 프롬프트에 "검토 항목"과 "출력 형식"을 명시하세요.

In [16]:
# TODO: Reflection Node 구현

MAX_REFLECTIONS = 1  # 최대 Reflection 반복 횟수 (무한 루프 방지)

def reflection_node(state: AgentState) -> AgentState:
    """
    Reflection Node: Worker의 결과를 검토하고 개선 제안을 합니다.
    
    Args:
        state: 현재 AgentState (result 포함)
    
    Returns:
        업데이트된 AgentState (reflection, final_result 필드 추가)
    """
    result = state["result"]
    user_request = state["user_request"]
    plan = state["plan"]
    
    # TODO: Reflection 프롬프트를 작성하세요
    # - Worker의 결과를 검토하고 원본 요청 충족 여부 평가
    # - 검토 항목: 요구사항 충족, 누락 정보, 정확성, 개선 필요 사항
    # - 출력 형식: ## 검토 결과 + ## 최종 개선 결과
    reflection_prompt = f"""
당신은 결과를 검토하는 Reflection Agent입니다.

Worker Agent가 생성한 결과를 검토하고, 원본 요청을 충족하는지 평가하세요.

검토 항목:
1. 원본 요청의 모든 요구사항이 충족되었는가?
2. 누락된 정보가 있는가?
3. 정보의 정확성과 구체성은 적절한가?
4. 개선이 필요한 부분이 있는가?

---
원본 요청: {user_request}

실행된 계획:
{plan}

Worker의 결과:
{result}

---
위 내용을 검토하고 다음 형식으로 작성하세요:

## 검토 결과
- 충족 여부: [충족/부분 충족/미충족]
- 강점: [잘된 부분]
- 개선 필요: [부족한 부분]

## 최종 개선 결과
[Worker의 결과를 보완하여 최종 결과 작성]
"""
    
    # LLM 호출
    response = llm.invoke(reflection_prompt)
    reflection = response.content

    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Reflection",
        "receiver": "User",
        "content": reflection,
        "task_type": "reflection"
    })
    
    # TODO: State 업데이트하여 반환 (reflection_count 증가 포함)
    return {
        **state,
        "reflection": reflection,  # TODO: reflection 변수로 교체
        "final_result": reflection,  # TODO: reflection 변수로 교체
        "reflection_count": state.get("reflection_count", 0) + 1,
        "messages": messages
    }

def should_continue_reflection(state: AgentState) -> str:
    """Reflection 후 종료할지 Worker로 돌아갈지 판단"""
    count = state.get("reflection_count", 0)
    reflection = state.get("reflection", "")
    
    # 최대 반복 횟수 초과 시 종료
    if count >= MAX_REFLECTIONS:
        return "end"
    
    # "충족" 판정이면 종료, 아니면 Worker로 재실행
    if "충족 여부: 충족" in reflection:
        return "end"
    
    return "worker"

# TODO 완료 후 테스트
reflection_state = reflection_node(final_state)
if reflection_state['reflection'] is not None:
    print("=== Reflection Node 결과 ===")
    print(f"검토 결과:\n{reflection_state['reflection'][:800]}...")
else:
    print("=== 코드를 완성해주세요 ===")

=== Reflection Node 결과 ===
검토 결과:
## 검토 결과
- 충족 여부: 부분 충족
- 강점:
  - 전체 여정 작성 프로세스를 단계별(15단계)로 잘 구조화했고, 각 단계의 산출물과 소요시간을 명확히 제시함.
  - Step1(사용자 정보 수집) 질문 목록이 상세하고 실용적이며, 요약표 예시도 제공되어 다음 단계로 이어가기 쉬움.
  - 수정 루프, 예약 우선순위, 문서 포맷 등 실무적 고려사항을 포함해 계획 운영 측면에서 잘 짜여 있음.
- 개선 필요:
  - 원본 요청(3일 서울 여행 계획 — 맛집, 관광지, 예산 포함)을 바로 충족시키는 실제 3일 일정·맛집 목록·예산안이 제공되지 않음. Worker는 프로세스 시작만 제안했음.
  - 즉시 활용 가능한 샘플 일정이나 예산(또는 기본안)을 제시하면 사용자가 빠르게 판단하거나 수정 요청하기 쉬움.
  - 일부 비용 추정(예: 공항 이동, 식비, 숙박 등)의 예시값이 포함되면 Step6~Step7의 결과물을 예상할 수 있어 더 유용함.
  - 예약 우선순위(어떤 맛집/명소를 먼저 예약해야 하는지)와 긴급 연락처 등 안전정보가 샘플로 제공되면 신뢰도가 올라감.

## 최종 개선 결과
(Worker의 단계별 계획은 유지하되, 사용자가 바로 활용하거나 선택·수정할 수 있도록 기본 샘플안과 예산 예시를 추가합니다. 필요 시 맞춤화해 드리겠습니다.)

1) 간단한 확인/다음 단계 제안
- 지금 Step1 질문으로 사용자 정보를 받는 방식은 적절합니다. 다만 사용자가 바로 응답하기 전에 참고할 수 있도록 "기본 샘플 3일 일정 + 맛집 + 예산(3안)"을 먼저 제시합니다. 사용자가 세부사항(날짜, 인원, 숙...


### TODO 5: 성과 평가 함수 구현

- **요구사항**: Multi-Agent 시스템의 결과를 평가하는 함수를 구현합니다.
- **입력**: 
  - `state` (AgentState): 현재 상태 딕셔너리 (user_request, plan, result, reflection 포함)
- **출력**: 
  - `dict`: 평가 결과
    - `score` (int): 종합 점수 (0~100)
    - `feedback` (str): 전체 피드백
    - `details` (dict): 세부 평가 항목별 점수
- **평가 항목**:
  - Planner 평가: 계획의 완성도, 단계 적절성 (0~30점)
  - Worker 평가: 실행 품질, 정보 충실도 (0~40점)
  - 전체 평가: 사용자 요청 충족도 (0~30점)
- **예상 결과**: 
  - 종합 점수: 70~90점 범위 (Reflection 적용 후 품질 향상)
  - JSON 출력 예시:
    ```json
    {
      "planner_score": 25,
      "worker_score": 35,
      "overall_score": 25,
      "feedback": "계획이 체계적이며, 대부분의 요청 사항을 충족함"
    }
    ```
- **확인 방법**: `print(f"종합 점수: {evaluation['score']}/100")`로 점수 출력
- **힌트**: LLM에게 평가 기준과 State 정보를 제공하여 점수와 피드백을 생성합니다. JSON 파싱 시 `re.search()`로 JSON 부분만 추출합니다.

In [17]:
# TODO: 성과 평가 함수 구현
# 아래 함수를 완성하세요.

import json
import re

def evaluate_multi_agent(state: AgentState) -> dict:
    """
    Multi-Agent 시스템의 성과를 평가합니다.
    
    Args:
        state: 현재 AgentState (user_request, plan, result, reflection 포함)
    
    Returns:
        dict: {"score": int, "feedback": str, "details": dict}
    """

    # 평가 프롬프트 구성
    eval_prompt = f"""
당신은 Multi-Agent 시스템의 성과를 평가하는 평가자입니다.

## 평가 기준
1. Planner 평가 (0~30점): 계획의 완성도, 단계 구성의 적절성
2. Worker 평가 (0~40점): 실행 품질, 정보의 충실도와 정확성
3. 전체 평가 (0~30점): 사용자 요청 충족도, 협업 효율성

## 입력 정보
- 사용자 요청: {state['user_request']}
- Planner의 계획: {state['plan']}
- Worker의 결과: {state['result']}
- Reflection 검토: {state.get('reflection', '없음')}

## 출력 형식
반드시 아래 JSON 형식만 출력하세요. 다른 텍스트는 포함하지 마세요.
{{"planner_score": <0~30>, "worker_score": <0~40>, "overall_score": <0~30>, "feedback": "<전체 피드백>"}}
"""

    response = llm.invoke(eval_prompt)
    content = response.content

    # JSON 파싱 (마크다운 코드 블록 처리)
    try:
        # 마크다운 코드 블록에서 JSON 추출 (```json ... ``` 또는 ``` ... ```)
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', content)
        if json_match:
            json_str = json_match.group(1)
        else:
            # 코드 블록이 없으면 중괄호로 시작하는 JSON 추출
            json_match = re.search(r'\{[\s\S]*\}', content)
            json_str = json_match.group(0) if json_match else content

        result = json.loads(json_str)

        # TODO: result를 이용하여 점수를 합산하고 결과를 반영하세요
        total_score = result['planner_score'] + result['worker_score'] + result['overall_score']          # TODO: 세 항목의 점수를 합산하세요

        return {
            "score": total_score,
            "feedback": result['feedback'],
            "details": {
                "planner": result['planner_score'],
                "worker": result['worker_score'],
                "overall": result['overall_score']
            }
        }
    except Exception as e:
        return {"score": 0, "feedback": f"평가 실패: {str(e)}", "details": {}}

# 테스트
evaluation = evaluate_multi_agent(reflection_state)
if evaluation['feedback'] is not None:
    print("=== 성과 평가 결과 ===")
    print(f"종합 점수: {evaluation['score']}/100")
    print(f"세부 점수: Planner {evaluation['details'].get('planner', 0)}/30, Worker {evaluation['details'].get('worker', 0)}/40, 전체 {evaluation['details'].get('overall', 0)}/30")
    print(f"피드백: {evaluation['feedback']}")
else:
    print("=== 코드를 완성해주세요 ===")

=== 성과 평가 결과 ===
종합 점수: 68/100
세부 점수: Planner 26/30, Worker 22/40, 전체 20/30
피드백: 강점: Planner는 15단계로 매우 체계적으로 프로세스를 설계했고, Step별 산출물과 예상 소요시간을 구체적으로 제시해 실무적 활용도가 높습니다. Step1 질문 목록도 상세하고 실제 조사·일정 작성에 바로 연결될 수 있게 구성되어 있습니다. 개선점(Planner): 원본 요청(3일 서울 여행 일정·맛집·예산)을 즉시 보여줄 수 있는 '기본 샘플안'을 계획 단계에 포함하면 사용자 결정을 빠르게 도와줄 수 있습니다. 또한 비용 추정의 예시값(항공·교통·숙박의 평균값)을 Step6에 미리 예시로 제시하면 Step7 결과 예측이 쉬워집니다. 개선점(Worker): Worker는 Planner의 절차를 잘 이해하고 권한을 요청했으나, 실제 산출물(샘플 일정·맛집 리스트·예산안)을 즉시 제공하지 않아 사용자가 바로 판단하거나 피드백 주기 어려웠습니다. 초기 응답 단계에서 기본 샘플 3일 일정과 표준 예산(예: 절약/표준/프리미엄 3안)을 선제적으로 제시한 뒤 사용자의 세부정보로 맞춤화하는 방식이 권장됩니다. 또한 예약 우선순위(구체적 업체 예시)와 긴급 연락처 샘플을 초안에 포함하면 신뢰도가 올라갑니다. 권장 실행: Worker가 제안한 두 옵션(사용자 맞춤 진행 or 기본안 즉시 조사)을 명확히 제시하여 사용자가 선택하도록 하고, 선택 시 즉시 Step1 요약표 또는 기본 샘플안(3일 일정+맛집+3안 예산)을 제공하여 다음 단계 진행을 가속화하세요. 응답·수정 기한과 예약 대행에 필요한 최소정보도 명확히 안내하면 운영 효율성이 개선됩니다.


### Guided Build: Reflection 패턴 워크플로우 구성

이제 Planner → Worker → Reflection 순서로 워크플로우를 구성합니다.
Reflection은 `should_continue_reflection` 함수로 **조건 분기**합니다:
- 품질 충분 또는 최대 반복 도달 → END
- 개선 필요 → Worker로 재실행

In [18]:
# Reflection 패턴 워크플로우 구성
workflow_reflection = StateGraph(AgentState)

# 노드 추가
workflow_reflection.add_node("planner", planner_node)
workflow_reflection.add_node("worker", worker_node)
workflow_reflection.add_node("reflection", reflection_node)

# 엣지 연결: planner → worker → reflection
workflow_reflection.add_edge(START, "planner")
workflow_reflection.add_edge("planner", "worker")
workflow_reflection.add_edge("worker", "reflection")

# Reflection 후 조건 분기: END 또는 Worker 재실행
workflow_reflection.add_conditional_edges(
    "reflection",
    should_continue_reflection,
    {"end": END, "worker": "worker"}
)

# 컴파일
app_reflection = workflow_reflection.compile()

print("Reflection 패턴 워크플로우 구성 완료!")
print("흐름: planner → worker → reflection → (조건 분기) → END 또는 worker")

Reflection 패턴 워크플로우 구성 완료!
흐름: planner → worker → reflection → (조건 분기) → END 또는 worker


In [19]:
# Reflection 패턴 워크플로우 실행
def run_multi_agent_reflection(user_request: str) -> dict:
    """
    Reflection 패턴이 포함된 Multi-Agent 시스템 실행
    
    Args:
        user_request: 사용자 요청
    
    Returns:
        최종 상태를 담은 딕셔너리
    """
    print("🚀 Reflection Multi-Agent 시스템 시작\n")
    
    # 초기 상태 설정
    initial_state = {
        "user_request": user_request
    }
    
    # 워크플로우 실행
    print("📋 [Planner Agent] 계획 수립 중...")
    print("⚙️ [Worker Agent] 계획 실행 중...")
    print("🔍 [Reflection Agent] 결과 검토 중...")
    
    final_state = app_reflection.invoke(initial_state)
    
    print(f"\n✅ 실행 완료 (Reflection 횟수: {final_state['reflection_count']})\n")
    
    return final_state

# 테스트 실행
reflection_result = run_multi_agent_reflection("서울 2박3일 여행 계획 + 예산 50만원 이내")

print("=" * 50)
print("📊 Reflection 최종 결과")
print("=" * 50)
print(reflection_result["final_result"])

🚀 Reflection Multi-Agent 시스템 시작

📋 [Planner Agent] 계획 수립 중...
⚙️ [Worker Agent] 계획 실행 중...
🔍 [Reflection Agent] 결과 검토 중...

✅ 실행 완료 (Reflection 횟수: 1)

📊 Reflection 최종 결과
## 검토 결과
- 충족 여부: 부분 충족
- 강점:
  - Step 1(사용자 정보 확인)을 충실히 구현한 점 — 필요한 질문 항목을 광범위하고 논리적으로 잘 정리함.
  - 숙박·교통·식사·선호도·예약대행 등 실제 조사·계획 단계에 필요한 핵심 정보를 빠짐없이 포함함.
  - 응답 예시를 제공해 사용자 입력을 유도하는 점이 실용적임.
- 개선 필요:
  - 원본 요청(“서울 2박3일 여행 계획 + 예산 50만원 이내”)에 대한 실제 일정·비용안(샘플 일정, 비용 시뮬레이션 등)을 전혀 제시하지 않음 — 현재는 오직 정보를 수집하는 질문지만 있음.
  - 사용자가 바로 응답하기 힘들 때를 대비한 기본(assumption) 샘플안 제공이 없음. 사용자가 정보를 즉시 주지 않을 경우에도 참고할 수 있는 예시 일정/예산을 제시하면 더 유용함.
  - 질문지에 소요 예상 준비시간(전체 계획 제시까지 걸리는 시간)이나 이후 진행 방식(사용자 응답 후 각 Step 수행 예상 소요) 안내가 빠져있음.
  - 결제·개인정보(예약 대행 시 결제/계좌/카드 정보 요구 등)에 관한 안전/프라이버시 안내가 누락됨.

## 최종 개선 결과
아래는 Worker의 Step1 질문지를 보완한 개선안(수정된 질문지 + 즉시 제공 가능한 샘플 일정 및 예산안)입니다. 사용자가 바로 답변을 주지 못할 경우, 이 가정 샘플안을 기반으로 Step2~16을 진행할지 선택하게 할 수 있습니다.

1) 보완된 사용자 정보 확인(질문지 — 복사해서 사용 가능)
필수 질문
1) 출발지(거주 도시/역·공항)
2) 출발 예정일 / 귀가 예정일(정확한 날짜 또는 유연성)
3) 여행 인원(총 인원, 본인 포함 

### Test: Reflection 패턴 실행 결과 확인

**확인 기준**:
- [ ] Reflection Agent가 Worker의 결과를 검토했는가?
- [ ] 충족 여부 판정이 이루어졌는가?
- [ ] 개선이 필요한 경우 Worker가 재실행되었는가?
- [ ] 최종 결과에 개선 내용이 반영되었는가?

**체감 포인트**: "Reflection Agent가 자동으로 결과를 검토하고 품질을 높여주니, 수동 검토 없이도 신뢰할 수 있는 결과를 얻을 수 있구나!"

---
## 트러블슈팅 가이드

실습/과제 진행 중 자주 발생하는 오류와 해결 방법입니다.

### API 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `AuthenticationError` | API Key 미설정 또는 잘못된 키 | `.env` 파일에 `GMS_KEY` 확인 |
| `RateLimitError` | API 호출 한도 초과 | 잠시 대기 후 재실행 (1~2분) |
| `InvalidRequestError` | 토큰 제한 초과 | 입력 텍스트 길이 줄이기 또는 청킹 크기 조정 |

### 패키지 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `ModuleNotFoundError` | 패키지 미설치 | Step 1의 `%pip install` 셀 재실행 |
| `ImportError: cannot import name` | 버전 불일치 | `%pip install --upgrade <패키지>` |
| ChromaDB `sqlite3` 오류 | SQLite 버전 낮음 | `%pip install pysqlite3-binary` 후 커널 재시작 |

### 일반

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `NameError: name 'xxx' is not defined` | 이전 셀 미실행 | Step 1부터 순서대로 재실행 |
| 출력이 비어있음 | 환경 변수 로드 실패 | `.env` 파일 경로 확인 후 커널 재시작 |


---

## 실습 마무리

### 학습 내용 정리

이번 실습에서 배운 핵심 내용:

1. **단일 Agent의 한계**: 복잡한 작업에서 혼란, 누락, 품질 저하 발생
2. **Multi-Agent 대표 패턴**: Planner-Worker, Supervisor, Debate, Reflection, Hierarchical
3. **Planner-Worker 패턴**: 계획 수립과 실행을 분리하여 체계적 결과 도출
4. **Reflection 패턴**: 자기 검토 루프로 결과 품질 향상
5. **LangGraph StateGraph**: 상태 기반 워크플로우로 에이전트 간 협업 구현

### 학생용 자가 체크리스트

- [ ] 단일 Agent의 한계를 직접 체험했는가?
- [ ] Multi-Agent 대표 패턴 5가지를 설명할 수 있는가?
- [ ] LangGraph State를 정의하고 이해했는가?
- [ ] Planner Node가 계획을 수립하는 코드를 작성했는가?
- [ ] Worker Node가 계획을 실행하는 코드를 작성했는가?
- [ ] Reflection Node가 결과를 검토하고 개선하는 코드를 작성했는가?
- [ ] Reflection 패턴 전후 결과 품질 차이를 확인했는가?
- [ ] 성과 평가 함수로 결과 품질을 정량적으로 측정했는가?

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.